In [21]:
from pymatgen.core.surface import SlabGenerator
from pymatgen.core.structure import Structure
import numpy as np
import sys

ID = (1, 1, 1)
NNN = '242'
# Define quantas camadas serão fixas no meio 
n_bottom, n_middle, n_top = int(NNN[0]), int(NNN[1]), int(NNN[2])
N = n_bottom + n_middle + n_top  # Número total de camadas

# Carrega estrutura do bulk
bulk_structure = Structure.from_file("../../inputs/structure/CONTCAR")

# Gerar slab
slabgen = SlabGenerator(
    bulk_structure,
    miller_index=ID,
    min_slab_size=N,
    min_vacuum_size=4,
    in_unit_planes=True,
    center_slab=True
)
slab = slabgen.get_slabs()[0]
ortho_slab = slab.get_orthogonal_c_slab()

ortho_slab = slabgen.nonstoichiometric_symmetrized_slab(ortho_slab)[0]  

# Posição z normalizada (coordenada fracionária)
z_coords = [site.frac_coords[2] for site in ortho_slab.sites]
sorted_indices = np.argsort(z_coords)
total_sites = len(z_coords)

N_bulk = len(bulk_structure)

# Divide os sites por camada usando ordenação em z
bottom_indices = sorted_indices[:N_bulk * n_bottom]
middle_indices = sorted_indices[N_bulk * n_bottom : total_sites - N_bulk * n_top]
top_indices = sorted_indices[total_sites - N_bulk * n_top :]

# Inicializa flags como todos móveis
selective_dynamics = [[True, True, True] for _ in range(total_sites)]

# Aplica restrições nas camadas do meio
for i in middle_indices:
    selective_dynamics[i] = [False, False, False]

# Ordena os sites de acordo com a espécie (opcional)
desired_species_order = [str(el) for el in bulk_structure.composition.elements]
sites_ordered = []
selective_dynamics_ordered = []
for sp in desired_species_order:
    for site, sdyn in zip(ortho_slab.sites, selective_dynamics):
        if site.species_string == sp:
            sites_ordered.append(site)
            selective_dynamics_ordered.append(sdyn)
    
# Criar nova estrutura com campos de dinâmica seletiva
new_structure = Structure(
    ortho_slab.lattice,
    [site.specie for site in sites_ordered],
    [site.frac_coords for site in sites_ordered],
    coords_are_cartesian=False)
new_structure.add_site_property("selective_dynamics", selective_dynamics_ordered)

# Salvar POSCAR com a tag "Selective dynamics"
from pymatgen.io.vasp.inputs import Poscar
poscar = Poscar(new_structure, selective_dynamics=True)
poscar.write_file("../../inputs/structure/POSCAR")

In [22]:
from pymatgen.core.surface import SlabGenerator
from pymatgen.core.structure import Structure
import numpy as np
import sys

ID = (0, 0, 1)
NNN = '101'
# Define quantas camadas serão fixas no meio 
n_bottom, n_middle, n_top = int(NNN[0]), int(NNN[1]), int(NNN[2])
N = n_bottom + n_middle + n_top  # Número total de camadas
TERM = 0

# Carrega estrutura do bulk
bulk_structure = Structure.from_file("../../inputs/structure/CONTCAR")

# Gerar slab
slabgen = SlabGenerator(
    bulk_structure,
    miller_index=ID,
    min_slab_size=N,
    min_vacuum_size=4,
    in_unit_planes=True,
    center_slab=True
)
slab = slabgen.get_slabs()[0]
ortho_slab = slab.get_orthogonal_c_slab()

ortho_slab = slabgen.nonstoichiometric_symmetrized_slab(ortho_slab)[TERM]

# Posição z normalizada (coordenada fracionária)
z_coords = [site.frac_coords[2] for site in ortho_slab.sites]
sorted_indices = np.argsort(z_coords)
total_sites = len(z_coords)

N_bulk = len(bulk_structure)

# Divide os sites por camada usando ordenação em z
bottom_indices = sorted_indices[:N_bulk * n_bottom]
middle_indices = sorted_indices[N_bulk * n_bottom : total_sites - N_bulk * n_top]
top_indices = sorted_indices[total_sites - N_bulk * n_top :]

# Inicializa flags como todos móveis
selective_dynamics = [[True, True, True] for _ in range(total_sites)]

# Aplica restrições nas camadas do meio
for i in middle_indices:
    selective_dynamics[i] = [False, False, False]

# Ordena os sites de acordo com a espécie (opcional)
desired_species_order = [str(el) for el in bulk_structure.composition.elements]
sites_ordered = []
selective_dynamics_ordered = []
for sp in desired_species_order:
    for site, sdyn in zip(ortho_slab.sites, selective_dynamics):
        if site.species_string == sp:
            sites_ordered.append(site)
            selective_dynamics_ordered.append(sdyn)

# Criar nova estrutura com campos de dinâmica seletiva
new_structure = Structure(
    ortho_slab.lattice,
    [site.specie for site in sites_ordered],
    [site.frac_coords for site in sites_ordered],
    coords_are_cartesian=False
)
new_structure.add_site_property("selective_dynamics", selective_dynamics_ordered)

# Salvar POSCAR com a tag "Selective dynamics"
from pymatgen.io.vasp.inputs import Poscar
poscar = Poscar(new_structure, selective_dynamics=True)
poscar.write_file("../../inputs/structure/POSCAR")

In [39]:
from pymatgen.core.surface import SlabGenerator
from pymatgen.core.structure import Structure
import numpy as np
import sys

ID = (0, 1, 1)
NNN = '262'
# Define quantas camadas serão fixas no meio 
n_bottom, n_middle, n_top = (int(NNN[0])), int(NNN[1]), (int(NNN[2]))
N = n_bottom + n_middle + n_top  # Número total de camadas
TERM = 1
MOD = 2

# Carrega estrutura do bulk
bulk_structure = Structure.from_file("../../inputs/structure/CONTCAR")

# Gerar slab
slabgen = SlabGenerator(
    bulk_structure,
    miller_index=ID,
    min_slab_size=N,
    min_vacuum_size=4,
    in_unit_planes=True,
    center_slab=True
)
slab = slabgen.get_slabs()[0]
ortho_slab = slab.get_orthogonal_c_slab()

ortho_slab = slabgen.nonstoichiometric_symmetrized_slab(ortho_slab)[TERM]

disc_bottom, disc_top = 0, 0

if MOD == 1:
    ortho_slab.remove_sites([N*5 - 7])
    ortho_slab.remove_sites([0])
    disc_bottom, disc_top = 1, 1
    
elif MOD == 2:
    ortho_slab.remove_sites([N*5 - 3])
    ortho_slab.remove_sites([N*5 - 6])
    ortho_slab.remove_sites([2])
    ortho_slab.remove_sites([1])
    disc_bottom, disc_top = 2, 2
    
elif MOD == 3:
    ortho_slab.remove_sites([N*5 - 5])
    ortho_slab.remove_sites([2])
    disc_bottom, disc_top = 1, 1
    
elif MOD == 4:
    ortho_slab.remove_sites([N*5 - 2])
    ortho_slab.remove_sites([N*5 - 3])
    ortho_slab.remove_sites([N*5 - 4])
    ortho_slab.remove_sites([N*5 - 5])
    ortho_slab.remove_sites([5])
    ortho_slab.remove_sites([3])
    ortho_slab.remove_sites([2])
    ortho_slab.remove_sites([1])
    disc_bottom, disc_top = 0, 0
    
if MOD == 5:
    ortho_slab.remove_sites([N*5 - 10])
    ortho_slab.remove_sites([0])
    disc_bottom, disc_top = 0, 0

# Posição z normalizada (coordenada fracionária)
z_coords = [site.frac_coords[2] for site in ortho_slab.sites]
sorted_indices = np.argsort(z_coords)
total_sites = len(z_coords)

N_bulk = len(bulk_structure)

# Divide os sites por camada usando ordenação em z
bottom_indices = sorted_indices[:N_bulk * n_bottom - disc_bottom]
middle_indices = sorted_indices[N_bulk * n_bottom - disc_bottom : total_sites - (N_bulk * n_top - disc_top)]
top_indices = sorted_indices[total_sites - (N_bulk * n_top - disc_top) :]

# Inicializa flags como todos móveis
selective_dynamics = [[True, True, True] for _ in range(total_sites)]

# Aplica restrições nas camadas do meio
for i in middle_indices:
    selective_dynamics[i] = [False, False, False]

# Ordena os sites de acordo com a espécie (opcional)
desired_species_order = [str(el) for el in bulk_structure.composition.elements]
sites_ordered = []
selective_dynamics_ordered = []
for sp in desired_species_order:
    for site, sdyn in zip(ortho_slab.sites, selective_dynamics):
        if site.species_string == sp:
            sites_ordered.append(site)
            selective_dynamics_ordered.append(sdyn)

# Criar nova estrutura com campos de dinâmica seletiva
new_structure = Structure(
    ortho_slab.lattice,
    [site.specie for site in sites_ordered],
    [site.frac_coords for site in sites_ordered],
    coords_are_cartesian=False
)
new_structure.add_site_property("selective_dynamics", selective_dynamics_ordered)

# Salvar POSCAR com a tag "Selective dynamics"
from pymatgen.io.vasp.inputs import Poscar
poscar = Poscar(new_structure, selective_dynamics=True)
poscar.write_file("../../inputs/structure/POSCAR")

In [40]:
N

10

In [41]:
for i, site in enumerate(new_structure):
    print(i, site.species_string, site.frac_coords)

0 Cs [-2.92855617e-16 -1.31890445e+00  1.88414921e-01]
1 Cs [-4.03877919e-16 -1.81890445e+00  2.59843492e-01]
2 Cs [-5.14900221e-16 -2.31890445e+00  3.31272064e-01]
3 Cs [-6.25922524e-16 -2.81890445e+00  4.02700635e-01]
4 Cs [-7.36944826e-16 -3.31890445e+00  4.74129206e-01]
5 Cs [-8.47967129e-16 -3.81890445e+00  5.45557778e-01]
6 Cs [-9.58989431e-16 -4.31890445e+00  6.16986349e-01]
7 Cs [-1.07001173e-15 -4.81890445e+00  6.88414921e-01]
8 Cs [-1.18103404e-15 -5.31890445e+00  7.59843492e-01]
9 Cs [-1.29205634e-15 -5.81890445e+00  8.31272064e-01]
10 Pb [ 0.5        -1.31890445  0.25984349]
11 Pb [ 0.5        -1.81890445  0.33127206]
12 Pb [ 0.5        -2.31890445  0.40270064]
13 Pb [ 0.5        -2.81890445  0.47412921]
14 Pb [ 0.5        -3.31890445  0.54555778]
15 Pb [ 0.5        -3.81890445  0.61698635]
16 Pb [ 0.5        -4.31890445  0.68841492]
17 Pb [ 0.5        -4.81890445  0.75984349]
18 Br [ 0.5        -1.56890445  0.22412921]
19 Br [ 0.5        -1.06890445  0.22412921]
20 Br [-4.

In [42]:
import subprocess
from tempfile import NamedTemporaryFile
from pymatgen.io.cif import CifWriter
import os

def view_in_vesta(structure, vesta_path="/home/gpereira/Downloads/VESTA-gtk3/VESTA"):
    """Abre uma estrutura pymatgen diretamente no VESTA."""
    with NamedTemporaryFile(suffix=".cif", delete=False) as tmpfile:
        CifWriter(structure).write_file(tmpfile.name)
        tmpfile_path = tmpfile.name

    try:
        subprocess.Popen([vesta_path, tmpfile_path])
        print(f"Abrindo estrutura no VESTA: {tmpfile_path}")
    except FileNotFoundError:
        print(f"Erro: o executável '{vesta_path}' não foi encontrado no PATH.")

view_in_vesta(new_structure)

Abrindo estrutura no VESTA: /tmp/tmpxwx_554i.cif



(VESTA-gui:73581): Gtk-CRITICAL **: 11:32:57.683: gtk_box_gadget_distribute: assertion 'size >= 0' failed in GtkScrollbar

(VESTA-gui:73581): Gtk-CRITICAL **: 11:32:57.683: gtk_box_gadget_distribute: assertion 'size >= 0' failed in GtkScrollbar

(VESTA-gui:73581): Gtk-CRITICAL **: 11:32:57.683: gtk_box_gadget_distribute: assertion 'size >= 0' failed in GtkScrollbar

(VESTA-gui:73581): Gtk-CRITICAL **: 11:32:57.683: gtk_box_gadget_distribute: assertion 'size >= 0' failed in GtkScrollbar

(VESTA-gui:73581): Gtk-CRITICAL **: 11:32:57.683: gtk_box_gadget_distribute: assertion 'size >= 0' failed in GtkNotebook

(VESTA-gui:73581): Gtk-CRITICAL **: 12:56:09.305: gtk_box_gadget_distribute: assertion 'size >= 0' failed in GtkNotebook
